# karambola vs pykarambola: Center Mode Tests

Numerical comparison of pykarambola against karambola 2.0 (C++) across three
test cases using two asymmetric boxes:

- **Box A**: 2×2×2, centred at (0, 0, 0)
- **Box B**: 4×4×4, centred at (20, 0, 0)

| Case | Mesh(es) | Labels | `center` modes tested |
|------|----------|--------|----------------------|
| 1 | box_a and box_b individually | none | `None`, `reference_centroid` |
| 2 | combined mesh | none (treated as one body) | `None`, `reference_centroid` |
| 3 | combined mesh | label 1 = box_a, label 2 = box_b | `None`, `reference_centroid` |

karambola was run as:
```
karambola --no-labels <mesh.off>                       # ref_origin
karambola --no-labels --reference_centroid <mesh.off>  # ref_centroid
karambola --labels   <mesh.off>                        # per-label ref_origin
karambola --labels   --reference_centroid <mesh.off>   # per-label ref_centroid
```


In [1]:
import os
import numpy as np
import pykarambola
from pykarambola import minkowski_tensors, parse_off_file, LABEL_UNASSIGNED

print(f"pykarambola version: {pykarambola.__version__}")

OFF_DIR  = "../dataset/basic_shapes/additivity_tests"
KARA_DIR = "../dataset/karambola_results/basic_shapes/additivity_tests"


pykarambola version: 0.2.0


## Helper functions

In [2]:
# ── Mesh loader ──────────────────────────────────────────────────────────────
def triangulation_to_arrays(tri):
    """Extract (verts, faces, labels) numpy arrays from a Triangulation.

    Calls _consolidate_lists() to ensure the internal append-mode lists have
    been converted to arrays, then returns copies of the underlying arrays.
    Returns labels=None when no labels were assigned (all LABEL_UNASSIGNED).

    Note: this helper relies on private Triangulation attributes because the
    public API does not yet expose a to_arrays() method. A future release
    should add a public bridge between the file parsers and minkowski_tensors.
    """
    
    tri._consolidate_lists()
    verts  = tri._verts.copy()
    faces  = tri._faces.copy()
    labels = tri._labels.copy()
    if np.all(labels == LABEL_UNASSIGNED):
        return verts, faces, None
    return verts, faces, labels


def load_off(path, with_labels=False):
    """Load an .off file using pykarambola's parser and return arrays.

    Parameters
    ----------
    path : str
        Path to the .off file.
    with_labels : bool
        If True, read face labels from the 4th value after vertex indices
        (R G B label format used by karambola).

    Returns
    -------
    verts : (V, 3) ndarray
    faces : (F, 3) ndarray
    labels : (F,) ndarray or None
    """
    tri = parse_off_file(path, with_labels=with_labels)
    return triangulation_to_arrays(tri)


# ── karambola result parser ───────────────────────────────────────────────────
def _data_lines(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                rows.append(line.split())
    return rows

def parse_karambola_dir(dirpath):
    """Parse a karambola output directory.

    Returns a dict keyed by label (int) or 'ALL' -> flat tensor dict.
    Single-body runs (ALL label) are returned as {'ALL': {...}}.
    Per-label runs produce {1: {...}, 2: {...}, ...}.
    """
    raw = {}

    for row in _data_lines(os.path.join(dirpath, "w000_w100_w200_w300")):
        lab, val, name = row[0], float(row[1]), row[2]
        raw.setdefault(lab, {})[name] = val

    for row in _data_lines(os.path.join(dirpath, "w010_w110_w210_w310")):
        lab, name = row[0], row[4]
        raw.setdefault(lab, {})[name] = np.array([float(row[1]),
                                                   float(row[2]),
                                                   float(row[3])])

    for tname in ("w020", "w102", "w120", "w202", "w220", "w320"):
        for row in _data_lines(os.path.join(dirpath, tname)):
            lab, name = row[0], row[10]
            raw.setdefault(lab, {})[name] = (
                np.array([float(row[i]) for i in range(1, 10)]).reshape(3, 3)
            )

    result = {}
    for k, v in raw.items():
        result[int(k) if k != "ALL" else "ALL"] = v
    return result


# ── Comparison table ──────────────────────────────────────────────────────────
SCALAR_KEYS = ["w000", "w100", "w200", "w300"]
VECTOR_KEYS = ["w010", "w110", "w210", "w310"]
MATRIX_KEYS = ["w020", "w102", "w120", "w202", "w220", "w320"]

def compare(kara, pykar, title="", tol=1e-6):
    if title:
        print(f"{'='*70}\n  {title}\n{'='*70}")
    print(f"  {'tensor':<10}  {'karambola':>18}  {'pykarambola':>18}  {'|Δ|_max':>12}")
    print(f"  {'-'*64}")
    overall = True
    for k in SCALAR_KEYS + VECTOR_KEYS + MATRIX_KEYS:
        if k not in kara or k not in pykar:
            continue
        kv = np.asarray(kara[k]);  pv = np.asarray(pykar[k])
        d = float(np.max(np.abs(kv - pv)))
        ok = "✓" if d < tol else "✗"
        if d >= tol:
            overall = False
        if kv.ndim == 0:
            print(f"  {k:<10}  {float(kv):>18.8g}  {float(pv):>18.8g}  {d:>12.3e}  {ok}")
        elif kv.ndim == 1:
            ks = np.array2string(kv, precision=4, suppress_small=True)
            ps = np.array2string(pv, precision=4, suppress_small=True)
            print(f"  {k:<10}  {ks:>18}  {ps:>18}  {d:>12.3e}  {ok}")
        else:
            kd = np.array2string(np.diag(kv), precision=4, suppress_small=True)
            pd = np.array2string(np.diag(pv), precision=4, suppress_small=True)
            print(f"  {k:<10}  {'diag'+kd:>18}  {'diag'+pd:>18}  {d:>12.3e}  {ok}")
    print(f"  Overall: {'PASS ✓' if overall else 'FAIL ✗'}\n")
    return overall


---
## Case 1 — Single bodies

Each box is loaded individually and run through pykarambola without labels.
karambola was run on the same individual `.off` files with `--no-labels`.


In [3]:
vA, fA, _ = load_off(f"{OFF_DIR}/box_a_2x2x2.off")
vB, fB, _ = load_off(f"{OFF_DIR}/box_b_4x4x4.off")

kara_a_origin   = parse_karambola_dir(f"{KARA_DIR}/box_a_2x2x2_ref_origin")["ALL"]
kara_a_centroid = parse_karambola_dir(f"{KARA_DIR}/box_a_2x2x2_ref_centroid")["ALL"]
kara_b_origin   = parse_karambola_dir(f"{KARA_DIR}/box_b_4x4x4_ref_origin")["ALL"]
kara_b_centroid = parse_karambola_dir(f"{KARA_DIR}/box_b_4x4x4_ref_centroid")["ALL"]

py_a_origin   = minkowski_tensors(vA, fA, center=None)
py_a_centroid = minkowski_tensors(vA, fA, center='reference_centroid')
py_b_origin   = minkowski_tensors(vB, fB, center=None)
py_b_centroid = minkowski_tensors(vB, fB, center='reference_centroid')

compare(kara_a_origin,   py_a_origin,   "Case 1 — box_a, center=None")
compare(kara_a_centroid, py_a_centroid, "Case 1 — box_a, center='reference_centroid'")
compare(kara_b_origin,   py_b_origin,   "Case 1 — box_b, center=None")
compare(kara_b_centroid, py_b_centroid, "Case 1 — box_b, center='reference_centroid'")


  Case 1 — box_a, center=None
  tensor               karambola         pykarambola       |Δ|_max
  ----------------------------------------------------------------
  w000                         8                   8     0.000e+00  ✓
  w100                         8                   8     0.000e+00  ✓
  w200                 6.2831853           6.2831853     4.130e-13  ✓
  w300                 4.1887902           4.1887902     3.610e-12  ✓
  w010                [0. 0. 0.]          [0. 0. 0.]     0.000e+00  ✓
  w110                [0. 0. 0.]          [0. 0. 0.]     0.000e+00  ✓
  w210             [-0. -0. -0.]          [0. 0. 0.]     1.110e-16  ✓
  w310                [0. 0. 0.]       [-0. -0.  0.]     7.494e-16  ✓
  w020        diag[2.6667 2.6667 2.6667]  diag[2.6667 2.6667 2.6667]     3.333e-12  ✓
  w102        diag[2.6667 2.6667 2.6667]  diag[2.6667 2.6667 2.6667]     3.333e-12  ✓
  w120        diag[4.4444 4.4444 4.4444]  diag[4.4444 4.4444 4.4444]     4.444e-12  ✓
  w202        diag

True

---
## Case 2 — Combined mesh, no labels (single body)

Both boxes are in one `.off` file. karambola was run with `--no-labels`,
treating the full mesh as one body. pykarambola is called with `labels=None`.


In [4]:
vAB, fAB, _ = load_off(f"{OFF_DIR}/asymmetric_two_labels.off")

kara_ab_origin   = parse_karambola_dir(f"{KARA_DIR}/asymmetric_two_labels_nolabels_ref_origin")["ALL"]
kara_ab_centroid = parse_karambola_dir(f"{KARA_DIR}/asymmetric_two_labels_nolabels_ref_centroid")["ALL"]

py_ab_origin   = minkowski_tensors(vAB, fAB, labels=None, center=None)
py_ab_centroid = minkowski_tensors(vAB, fAB, labels=None, center='reference_centroid')

compare(kara_ab_origin,   py_ab_origin,   "Case 2 — combined mesh (no labels), center=None")
compare(kara_ab_centroid, py_ab_centroid, "Case 2 — combined mesh (no labels), center='reference_centroid'")


  Case 2 — combined mesh (no labels), center=None
  tensor               karambola         pykarambola       |Δ|_max
  ----------------------------------------------------------------
  w000                        72                  72     2.842e-14  ✓
  w100                        40                  40     0.000e+00  ✓
  w200                 18.849556           18.849556     3.876e-11  ✓
  w300                 8.3775804           8.3775804     2.780e-12  ✓
  w010        [1280.    0.    0.]  [1280.    0.    0.]     0.000e+00  ✓
  w110          [640.   0.   0.]    [640.   0.   0.]     1.137e-13  ✓
  w210        [251.3274  -0.      -0.    ]  [251.3274   0.       0.    ]     1.834e-10  ✓
  w310        [83.7758  0.      0.    ]  [83.7758 -0.      0.    ]     2.778e-11  ✓
  w020        diag[25688.    88.    88.]  diag[25688.    88.    88.]     3.638e-12  ✓
  w102        diag[13.3333 13.3333 13.3333]  diag[13.3333 13.3333 13.3333]     3.333e-11  ✓
  w120        diag[12875.5556    75.5556  

True

---
## Case 3 — Combined mesh, per-label

Same combined mesh. karambola was run with `--labels`, producing separate
results for label 1 (box_a) and label 2 (box_b). pykarambola is called with
the face label array read directly from the `.off` file.


In [5]:
vAB, fAB, labAB = load_off(f"{OFF_DIR}/asymmetric_two_labels.off", with_labels=True)
print(f"Labels in mesh: {np.unique(labAB)}  "
      f"(label 1: {(labAB==1).sum()} faces, label 2: {(labAB==2).sum()} faces)")

kara_lbl_origin   = parse_karambola_dir(f"{KARA_DIR}/asymmetric_two_labels_labels_ref_origin")
kara_lbl_centroid = parse_karambola_dir(f"{KARA_DIR}/asymmetric_two_labels_labels_ref_centroid")

py_lbl_origin   = minkowski_tensors(vAB, fAB, labels=labAB, center=None)
py_lbl_centroid = minkowski_tensors(vAB, fAB, labels=labAB, center='reference_centroid')


Labels in mesh: [1 2]  (label 1: 12 faces, label 2: 12 faces)


In [6]:
for lab in [1, 2]:
    compare(kara_lbl_origin[lab],   py_lbl_origin[lab],
            f"Case 3 — label {lab}, center=None")
    compare(kara_lbl_centroid[lab], py_lbl_centroid[lab],
            f"Case 3 — label {lab}, center='reference_centroid'")


  Case 3 — label 1, center=None
  tensor               karambola         pykarambola       |Δ|_max
  ----------------------------------------------------------------
  w000                         8                   8     0.000e+00  ✓
  w100                         8                   8     0.000e+00  ✓
  w200                 6.2831853           6.2831853     4.130e-13  ✓
  w300                 4.1887902           4.1887902     3.610e-12  ✓
  w010                [0. 0. 0.]          [0. 0. 0.]     0.000e+00  ✓
  w110                [0. 0. 0.]          [0. 0. 0.]     0.000e+00  ✓
  w210             [-0. -0. -0.]          [0. 0. 0.]     1.110e-16  ✓
  w310                [0. 0. 0.]       [-0. -0.  0.]     7.494e-16  ✓
  w020        diag[2.6667 2.6667 2.6667]  diag[2.6667 2.6667 2.6667]     3.333e-12  ✓
  w102        diag[2.6667 2.6667 2.6667]  diag[2.6667 2.6667 2.6667]     3.333e-12  ✓
  w120        diag[4.4444 4.4444 4.4444]  diag[4.4444 4.4444 4.4444]     4.444e-12  ✓
  w202        di

---
## Summary

In [7]:
results = {
    "Case 1 — box_a,          center=None":              (kara_a_origin,   py_a_origin),
    "Case 1 — box_a,          center=reference_centroid":(kara_a_centroid, py_a_centroid),
    "Case 1 — box_b,          center=None":              (kara_b_origin,   py_b_origin),
    "Case 1 — box_b,          center=reference_centroid":(kara_b_centroid, py_b_centroid),
    "Case 2 — combined/nolbl, center=None":              (kara_ab_origin,  py_ab_origin),
    "Case 2 — combined/nolbl, center=reference_centroid":(kara_ab_centroid,py_ab_centroid),
    "Case 3 — label 1,        center=None":              (kara_lbl_origin[1],   py_lbl_origin[1]),
    "Case 3 — label 1,        center=reference_centroid":(kara_lbl_centroid[1], py_lbl_centroid[1]),
    "Case 3 — label 2,        center=None":              (kara_lbl_origin[2],   py_lbl_origin[2]),
    "Case 3 — label 2,        center=reference_centroid":(kara_lbl_centroid[2], py_lbl_centroid[2]),
}

TOL = 1e-6
print(f"  {'Test':<52}  {'Max |Δ|':>12}  {'Pass?'}")
print(f"  {'-'*72}")
for desc, (k, p) in results.items():
    diffs = []
    for key in SCALAR_KEYS + VECTOR_KEYS + MATRIX_KEYS:
        if key in k and key in p:
            diffs.append(float(np.max(np.abs(np.asarray(k[key]) - np.asarray(p[key])))))
    max_d = max(diffs) if diffs else float('nan')
    ok = "✓" if max_d < TOL else "✗"
    print(f"  {desc:<52}  {max_d:>12.3e}  {ok}")


  Test                                                       Max |Δ|  Pass?
  ------------------------------------------------------------------------
  Case 1 — box_a,          center=None                     4.444e-12  ✓
  Case 1 — box_a,          center=reference_centroid       4.444e-12  ✓
  Case 1 — box_b,          center=None                     3.333e-08  ✓
  Case 1 — box_b,          center=reference_centroid       1.834e-10  ✓
  Case 2 — combined/nolbl, center=None                     4.445e-08  ✓
  Case 2 — combined/nolbl, center=reference_centroid       4.814e-09  ✓
  Case 3 — label 1,        center=None                     4.444e-12  ✓
  Case 3 — label 1,        center=reference_centroid       4.444e-12  ✓
  Case 3 — label 2,        center=None                     3.333e-08  ✓
  Case 3 — label 2,        center=reference_centroid       1.834e-10  ✓
